In [8]:
import pandas as pd
df=pd.read_csv("Retail cleaned.csv",header=None)
df.columns = ['Invoice','StockCode','Description','Quantity',
              'InvoiceDate','Price','Customer_ID','Sourcesheet']
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()


C:\Users\Ashish Sharma\AppData\Local\Temp\ipykernel_26904\3768650288.py:2: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("Retail cleaned.csv",header=None)


(1033034, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1033034 entries, 0 to 1033033
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033034 non-null  object        
 1   StockCode    1033034 non-null  str           
 2   Description  1028759 non-null  str           
 3   Quantity     1033034 non-null  int64         
 4   InvoiceDate  1033034 non-null  datetime64[us]
 5   Price        1033034 non-null  float64       
 6   Customer_ID  797883 non-null   float64       
 7   Sourcesheet  1033034 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 103.2+ MB


In [9]:
 # FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY

import pandas as pd

# Filter valid customer records and positive quantities
rfm_data = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

# Convert InvoiceDate into datetime
rfm_data['InvoiceDate'] = pd.to_datetime(
    rfm_data['InvoiceDate'], errors='coerce'
)

# Set snapshot date
snapshot_date = pd.to_datetime('2011-12-10')

# Calculate RFM
rfm = rfm_data.groupby('Customer_ID').agg(

    # Recency: Days since last purchase
    Recency=('InvoiceDate',
             lambda x: (snapshot_date - x.max()).days),

    # Frequency: Number of unique invoices
    Frequency=('Invoice', 'nunique'),

    # Monetary: Total customer spending
    Monetary=('Quantity',
              lambda x: (
                  x * rfm_data.loc[x.index, 'Price']
              ).sum())

).reset_index()

# Display result
print(rfm.head())

# Check information
rfm.info()

   Customer_ID  Recency  Frequency  Monetary
0      12346.0      326         12  77556.46
1      12347.0        3          8   4921.53
2      12348.0       76          5   2019.40
3      12349.0       19          4   4428.69
4      12350.0      311          1    334.40
<class 'pandas.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer_ID  5881 non-null   float64
 1   Recency      5881 non-null   int64  
 2   Frequency    5881 non-null   int64  
 3   Monetary     5881 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 183.9 KB


In [10]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

# 1. Calculate RFM Scores

rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# 2. Assign Customer Segments

def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


# 3. Apply Customer Segmentation

rfm['Segment'] = rfm.apply(segment_customer, axis=1)


# 4. Identify Churned Customers

rfm['IsChurned'] = rfm['Recency'] > 90


# 5. Display Final Results

print(rfm.head(10))

# Check customer segments
print(rfm['Segment'].value_counts())

   Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0      12346.0      326         12  77556.46        2        5        5   
1      12347.0        3          8   4921.53        5        4        5   
2      12348.0       76          5   2019.40        3        4        4   
3      12349.0       19          4   4428.69        5        3        5   
4      12350.0      311          1    334.40        2        1        2   
5      12351.0      376          1    300.93        2        1        2   
6      12352.0       37         10   2849.84        4        5        4   
7      12353.0      205          2    406.76        2        2        2   
8      12354.0      233          1   1079.40        2        1        3   
9      12355.0      215          2    947.61        2        2        3   

     Segment  IsChurned  
0    At Risk       True  
1  Champions      False  
2      Loyal      False  
3      Loyal      False  
4       Lost       True  
5       Lost      

In [11]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

# 1. Filter valid customer records
df = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

# 2. Convert InvoiceDate into datetime
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'], errors='coerce'
)

# 3. Create Invoice Month
df['InvoiceMonth'] = (
    df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
)

# 4. Calculate Revenue
df['Revenue'] = df['Quantity'] * df['Price']

# 5. Merge RFM results with transaction data
df_final = df.merge(
    rfm[
        [
            'Customer_ID',
            'Recency',
            'Frequency',
            'Monetary',
            'R_Score',
            'F_Score',
            'M_Score',
            'Segment',
            'IsChurned'
        ]
    ],
    on='Customer_ID',
    how='left'
)

# 6. Export final dataset to Excel
df_final.to_excel(
    'EcommerceDashboardData.xlsx',
    index=False
)

# 7. Display final results
print(df_final.head())

print("Final Shape:", df_final.shape)

print("Excel file exported successfully!")

  Invoice StockCode                         Description  Quantity InvoiceDate  \
0  524201     21017      RED BIRD HOUSE TREE DECORATION        24  2010-09-28   
1  524201     21431  GREEN GINGHAM FLOWER JEWELLERY BOX         8  2010-09-28   
2  524201     22034                ROBIN CHRISTMAS CARD        36  2010-09-28   
3  524201     22044    CHRISTMAS CARD STACK OF PRESENTS        36  2010-09-28   
4  524201     22139   RETRO SPOT TEA SET CERAMIC 11 PC         12  2010-09-28   

   Price  Customer_ID Sourcesheet InvoiceMonth  Revenue  Recency  Frequency  \
0   0.85      17863.0   2010-2011   2010-09-01    20.40       60         12   
1   1.65      17863.0   2010-2011   2010-09-01    13.20       60         12   
2   0.42      17863.0   2010-2011   2010-09-01    15.12       60         12   
3   0.42      17863.0   2010-2011   2010-09-01    15.12       60         12   
4   4.95      17863.0   2010-2011   2010-09-01    59.40       60         12   

   Monetary  R_Score  F_Score  M_Score